There are 3 common correlation analysis methods:
- Pearson correlation: 
  - measures linear correlation **between two continuous variables**.
  - It's the most common method. However, it assumes normality and linearity. Homoscedasticity is also assumed.
- Spearman correlation: 
  - measures **monotonic** relationship between two variables **using ranks**.
- Kendall correlation: 
  - measures ordinal association between two variables based on concordant and discordant pairs.
  - It's also using ranks. The ranking method is more robust to small sample sizes and many **tied** ranks.

Many python libraries can compute these correlations, such as `pandas`, `scipy.stats`, and `statsmodels`. Here we mainly use `pandas` and `scipy.stats`.

## Import Libraries

## Data Generation

We create a synthetic dataset with 4 continuous variables and 2 categorical variables (one with many tied value, the other not) for demonstration.

TODO: Try with real data, e.g., PANSS scores and cytokine levels.

## Methods Demonstration

We demonstrate the 3 correlation methods using `pandas` and `scipy.stats`.
We can compare the results from different methods and libraries.

In [ ]:
# Using scipy.stats

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau, pearsonr

# Spearman
rho, p_s = spearmanr(panss, cytokine)

# Kendall
tau, p_k = kendalltau(panss, cytokine)

# Pearson on log-transformed cytokine
log_cyt = np.log10(cytokine)
r, p_p = pearsonr(panss, log_cyt)

print(rho, tau, r)

# When to use scipy.stats:
# - When we need p-values for FDR correction

In [ ]:
# Using pandas

# Advantage: easier to use with DataFrames
# If we don't need p-values, pandas is more convenient
# If we need correlation matrix between multiple variables, pandas is handy.

## Best methos for different scenarios

- For two continuous variables with linear relationship: Pearson correlation is preferred.
- For two continuous variables with monotonic but non-linear relationship: Spearman or Kendall correlation is better.
- For ordinal variables or variables with many tied ranks: Kendall correlation is more robust.

(TODO: I want to show why with examples and visualizations.)

## Deeper Analysis

- Quantile regression (PANSS vs cytokine)
- Generalized additive models (GAMs)
- Bayesian correlation models
- Latent inflammation scores (PCA/PLS)

How I’d choose among these (quick guidance)
	•	You suspect “only high cytokine people matter” → Quantile regression
	•	You suspect nonlinearity/thresholds → GAM
	•	You want robust inference + uncertainty, not p-values → Bayesian robust regression
	•	You have many cytokines and want a single biology-driven axis → PCA (exploration) + PLS (prediction)

1) Quantile regression (PANSS vs cytokine)

What it answers:
Does PANSS relate differently to low vs high cytokine values? (e.g., effect only in the inflamed subgroup)

Why it’s good here:
	•	Robust to outliers compared with OLS
	•	Models heteroscedasticity naturally
	•	Great when the relationship is present only in upper tail (common in inflammation)

Typical setup: model log(cytokine) as outcome, PANSS as predictor, run multiple quantiles.


Interpretation:
If slope is near 0 at q=0.25 but positive at q=0.90 → PANSS is mainly associated with high inflammation individuals.

Upgrade: add covariates the same way: log_cyt ~ panss + age + sex + bmi + smoking + meds.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.DataFrame({"panss": panss, "cyt": cytokine})
df["log_cyt"] = np.log10(df["cyt"])

for q in [0.25, 0.50, 0.75, 0.90]:
    mod = smf.quantreg("log_cyt ~ panss", df).fit(q=q)
    print(q, mod.params["panss"], mod.pvalues["panss"])

2) Generalized Additive Models (GAMs)

What it answers:
Is the relationship nonlinear (thresholds, plateaus) rather than monotonic/linear?

Why it’s good here:
	•	Fits a smooth function f(\text{PANSS}) instead of forcing a straight line
	•	Very readable plots (“shape of effect”)
	•	Works well on log-transformed cytokines

Typical setup: log(cytokine) ~ s(PANSS) + covariates

Use `pygam`


Interpretation:
	•	If the smooth is flat until PANSS ~ 40 and rises after → potential symptom severity threshold.
	•	Report: EDF (effective degrees of freedom), p-value for smooth term, and plot.

Pitfall: GAMs can **overfit** if you allow too much wiggle. Keep smoothing reasonable and **always plot**.

In [ ]:
import numpy as np
from pygam import LinearGAM, s

X = np.asarray(panss).reshape(-1, 1)
y = np.log10(np.asarray(cytokine))

gam = LinearGAM(s(0)).fit(X, y)
print(gam.summary())

# To visualize the smooth effect:
XX = gam.generate_X_grid(term=0)
pred = gam.predict(XX)

Bayesian correlation models

What it answers:
What’s the posterior distribution of association (uncertainty fully quantified), and how strong is evidence for positive/negative association?

Why it’s good here:
	•	Gives credible intervals (not just p-values)
	•	You can model heavy tails/outliers explicitly (Student-t)
	•	Natural for small-to-moderate n and noisy biomarkers

Practical recommendation

Instead of a “Bayesian Pearson on raw cytokine”, do a Bayesian regression on log(cytokine) with Student-t errors (robust) and interpret slope as association.

Use `pymc3` or `stan` for implementation.

Interpretation you’ll report:
	•	Posterior mean of β
	•	95% credible interval
	•	P(\beta > 0) (or P(\beta < 0))
	•	Optional: Bayes factor (if you go that route)

Key benefit: you can say “there’s a 97% posterior probability the association is positive,” which is often clearer than p-values.

In [ ]:
import numpy as np
import pymc as pm

x = (np.asarray(panss) - np.mean(panss)) / np.std(panss)
y = np.log10(np.asarray(cytokine))

with pm.Model() as model:
    alpha = pm.Normal("alpha", 0, 1)
    beta  = pm.Normal("beta", 0, 1)          # association
    sigma = pm.HalfNormal("sigma", 1)
    nu    = pm.Exponential("nu", 1/10) + 1   # df for Student-t

    mu = alpha + beta * x
    obs = pm.StudentT("obs", nu=nu, mu=mu, sigma=sigma, observed=y)

    idata = pm.sample(2000, tune=2000, target_accept=0.9)

# Summarize posterior of beta: mean, 95% credible interval, P(beta>0)

4) Latent inflammation scores (PCA / PLS)

This is the right move if you have multiple cytokines and you suspect a shared “inflammation axis”.

4A) PCA inflammation score (unsupervised)

What it answers:
Is there a dominant latent inflammatory pattern, and does PANSS relate to that pattern?

Why it’s good:
	•	Reduces multiple testing
	•	Captures correlated cytokine structure
	•	Good exploratory + stable summary score

Setup tips:
	•	Log-transform cytokines
	•	Standardize (z-score)
	•	PCA → use PC1 (often “overall inflammation”), then correlate/regress with PANSS


Then:
	•	Spearman( PANSS, PC1 ) + CI
	•	Or regression: PC1 ~ PANSS + covariates

Interpretation:
Loadings tell you which cytokines define the axis. PC1 can reflect “pro-inflammatory mix” vs “anti-inflammatory mix” depending on signs.


In [ ]:
# 4A

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# cytokines_matrix: shape (n_samples, n_cytokines)
Y = np.log10(cytokines_matrix)

Yz = StandardScaler().fit_transform(Y)
pca = PCA(n_components=3).fit(Yz)
scores = pca.transform(Yz)

inflam_pc1 = scores[:, 0]  # latent inflammation score

4B) PLS (supervised)

What it answers:
What linear combination of cytokines best predicts PANSS (or best covaries with it)?

Why it’s good:
	•	Uses PANSS signal to define the component (unlike PCA)
	•	Often higher sensitivity if a specific cytokine pattern relates to symptoms



Crucial: use cross-validation to pick n_components and avoid overfitting.

Interpretation:
PLS weights identify a cytokine signature most aligned with PANSS; then you report predictive performance (CV R²) and stability.

In [ ]:
# 4B

import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(np.log10(cytokines_matrix))
y = np.asarray(panss)

pls = PLSRegression(n_components=1)
pls.fit(X, y)

# Component scores (latent inflammation pattern predictive of PANSS)
pls_score = pls.x_scores_[:, 0]
weights = pls.x_weights_[:, 0]